<a href="https://colab.research.google.com/github/rangaraju1/AI-agents/blob/main/HealthSystemFinder_vector_db.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem Statement: Symptom Finder for HealthEase!

### You've been hired by HealthEase an advanced clinic where AI helps doctors make faster and smarter decisions. You've been tasked with building a system that can assist doctors by finding similar past cases based on symptoms.

# Problem Statement: Symptom Finder for HealthEase!

### You've been hired by HealthEase an advanced clinic where AI helps doctors make faster and smarter decisions. You've been tasked with building a system that can assist doctors by finding similar past cases based on symptoms.

# Problem Statement: Symptom Finder for HealthEase!

### You've been hired by HealthEase an advanced clinic where AI helps doctors make faster and smarter decisions. You've been tasked with building a system that can assist doctors by finding similar past cases based on symptoms.

Import ChromaDB

In [ ]:
# Install ChromaDB if running in Google Colab
!pip install chromadb -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 1.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

In [ ]:
import chromadb
import csv

Instantiate a chromadb client and create the `medical_notes` collection.

> **Fix 1:** Use `EphemeralClient()` instead of the deprecated `Client()`.
>
> **Fix 2:** Use `get_or_create_collection()` so re-running the notebook doesn't raise a `UniqueConstraintError`.

In [ ]:
client = chromadb.EphemeralClient()
collection = client.get_or_create_collection("medical_notes")

Implement the `ingest_notes` function to load the CSV and ingest each row (symptoms are ingested as docs and each row is ingested as metadata).

> **Fix 3:** Added the missing `collection.add()` call. All arguments must be lists, even for a single record.
>
> **Fix 4:** Used `collection.upsert()` instead of `collection.add()` to gracefully handle duplicate `patient_id` values in the CSV.

In [ ]:
def ingest_notes(csv_path: str):
    with open(csv_path, "r") as file:
        reader = csv.DictReader(file)
        for row in reader:
            patient_id = row["patient_id"]
            symptoms = row["symptoms"]
            # upsert handles duplicate patient_id rows without raising an error
            collection.upsert(
                documents=[symptoms],
                ids=[patient_id],
                metadatas=[row]
            )

Implement the `query_notes` function to fetch and print relevant patient records from chromadb. The records should be printed in the following format:

```text
Result 1:
Symptoms: Sharp back pain, numbness in legs
Diagnosis: Herniated Disc
Treatment: Physical therapy, pain management
Doctor Notes: MRI suggested to confirm disc herniation.
```

> **Fix 5:** Implemented the missing `query_notes()` function body.

In [ ]:
def query_notes(symptoms: str, top_k: int = 5):
    results = collection.query(
        query_texts=[symptoms],
        n_results=top_k
    )
    for i, meta in enumerate(results["metadatas"][0]):
        print(f"Result {i+1}:")
        print(f"Symptoms: {meta['symptoms']}")
        print(f"Diagnosis: {meta['diagnosis']}")
        print(f"Treatment: {meta['treatment']}")
        print(f"Doctor Notes: {meta['doctor_notes']}")
        print()

Upload your `healthcare_patient_records.csv` to Colab using the files panel on the left, then run the cells below.

> **Fix 6:** Replaced the `'PATH_TO_CSV'` placeholder with the actual filename.

In [ ]:
ingest_notes('healthcare_patient_records.csv')

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 66.4MiB/s]


In [ ]:
query_notes('Sharp back pain and numbness in legs')

Result 1:
Symptoms: Sharp back pain, numbness in legs
Diagnosis: Herniated Disc
Treatment: Physical therapy, pain management
Doctor Notes: MRI suggested to confirm disc herniation.

Result 2:
Symptoms: Sharp back pain, numbness in legs
Diagnosis: Herniated Disc
Treatment: Physical therapy, pain management
Doctor Notes: MRI suggested to confirm disc herniation.

Result 3:
Symptoms: Sharp back pain, numbness in legs
Diagnosis: Herniated Disc
Treatment: Physical therapy, pain management
Doctor Notes: MRI suggested to confirm disc herniation.

Result 4:
Symptoms: Sharp back pain, numbness in legs
Diagnosis: Herniated Disc
Treatment: Physical therapy, pain management
Doctor Notes: MRI suggested to confirm disc herniation.

Result 5:
Symptoms: Sharp back pain, numbness in legs
Diagnosis: Herniated Disc
Treatment: Physical therapy, pain management
Doctor Notes: MRI suggested to confirm disc herniation.

